In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Tue Jul 21 17:41:22 2026

@author: tasha
"""

# ============================================================
# Download all data files (SVD neural data + behavior files)
# from Figshare to your local computer
# ============================================================

import os
import requests

# where the files will be saved
root = r"C:\Users\tasha\OneDrive\Desktop\Neuromatch Academy"
os.makedirs(root, exist_ok=True)

BASE_URL = 'https://api.figshare.com/v2'
DATASET_ID = 28811129  # the Zhong et al. 2025 dataset ID

# ------------------------------------------------------------
# The 4 behavior files (each holds multiple mice inside it)
# ------------------------------------------------------------
beh_files = [
    'Beh_sup_train1_before_learning.npy',
    'Beh_sup_train1_after_learning.npy',
    'Beh_unsup_train1_before_learning.npy',
    'Beh_unsup_train1_after_learning.npy',
]

# ------------------------------------------------------------
# All 13 mice's session IDs (before + after learning)
# ------------------------------------------------------------
mouse_sessions = [
    'VR2_2021_03_20_1', 'VR2_2021_04_06_1',
    'TX60_2021_04_10_1', 'TX60_2021_05_04_1',
    'TX108_2023_03_13_1', 'TX108_2023_03_22_1',
    'TX109_2023_03_27_1', 'TX109_2023_04_07_1',
    'TX83_2022_08_17_1', 'TX83_2022_08_29_1',
    'TX88_2022_06_13_2', 'TX88_2022_06_17_2',
    'TX105_2022_10_08_2', 'TX105_2022_10_19_2',
    'TX104_2022_10_12_1', 'TX104_2022_10_20_1',
    'TX119_2023_12_14_1', 'TX119_2023_12_23_1',
    'TX123_2023_12_21_1', 'TX123_2024_01_02_1',
    'DR10_2022_07_12_1', 'DR10_2022_07_19_1',
    'DR15_2022_10_09_1', 'DR15_2022_10_19_1',
    'TX85_2022_06_14_1', 'TX85_2022_06_17_1',
]

# ------------------------------------------------------------
# Get the file listing from Figshare so we know the exact
# download link (file ID) for each file we need
# ------------------------------------------------------------
print("Fetching file list from Figshare...")
r = requests.get(f"{BASE_URL}/articles/{DATASET_ID}")
r.raise_for_status()
file_metadata = r.json()

all_figshare_files = file_metadata['files']
print(f"Found {len(all_figshare_files)} files in the dataset.\n")


def download_file(filename, save_path):
    match = next((f for f in all_figshare_files if f['name'] == filename), None)
    if match is None:
        print(f"  WARNING: could not find '{filename}' in the dataset listing - skipping")
        return

    if os.path.isfile(save_path):
        print(f"  already have {filename}, skipping")
        return

    size_mb = match['size'] / 1e6
    print(f"  downloading {filename} ({size_mb:.1f} MB)...")
    response = requests.get(f"{BASE_URL}/file/download/{match['id']}")
    response.raise_for_status()
    with open(save_path, 'wb') as f:
        f.write(response.content)
    print(f"    done.")


# ------------------------------------------------------------
# Download the 4 behavior files
# ------------------------------------------------------------
print("Downloading behavior files...")
for fname in beh_files:
    download_file(fname, os.path.join(root, fname))

# ------------------------------------------------------------
# Download each mouse's SVD neural data file (this is the big one)
# ------------------------------------------------------------
print("\nDownloading SVD neural data files (this may take a while)...")
for session_id in mouse_sessions:
    fname = f"{session_id}_SVD_dec.npy"
    download_file(fname, os.path.join(root, fname))

# ------------------------------------------------------------
# Download the trans.npz files (needed later for area breakdown)
# ------------------------------------------------------------
print("\nDownloading trans.npz files (brain area info)...")
seen = set()
for session_id in mouse_sessions:
    trans_name = session_id[:-1] + 'trans.npz'
    if trans_name in seen:
        continue
    seen.add(trans_name)
    download_file(trans_name, os.path.join(root, trans_name))

print("\nAll downloads complete! Check your folder:")
print(root)

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Computes d' and running speed, using Rui's function, for ALL 13 MICE
(before and after learning each) - 26 sessions total.
"""

from pathlib import Path
import os
import gc
import numpy as np

# ------------------------------------------------------------
# Your folder with the downloaded data
# ------------------------------------------------------------
root = Path(r"C:\Users\tasha\OneDrive\Desktop\Neuromatch Academy")

# which mice are supervised vs unsupervised
supervised_mice = ['VR2', 'TX60', 'TX108', 'TX109']
unsupervised_mice = ['TX83', 'TX88', 'TX105', 'TX104', 'TX119', 'TX123', 'DR10', 'DR15', 'TX85']

# each mouse's (before-session-id, after-session-id)
mice_sessions = {
    'VR2':   ('VR2_2021_03_20_1',   'VR2_2021_04_06_1'),
    'TX60':  ('TX60_2021_04_10_1',  'TX60_2021_05_04_1'),
    'TX108': ('TX108_2023_03_13_1', 'TX108_2023_03_22_1'),
    'TX109': ('TX109_2023_03_27_1', 'TX109_2023_04_07_1'),
    'TX83':  ('TX83_2022_08_17_1',  'TX83_2022_08_29_1'),
    'TX88':  ('TX88_2022_06_13_2',  'TX88_2022_06_17_2'),
    'TX105': ('TX105_2022_10_08_2', 'TX105_2022_10_19_2'),
    'TX104': ('TX104_2022_10_12_1', 'TX104_2022_10_20_1'),
    'TX119': ('TX119_2023_12_14_1', 'TX119_2023_12_23_1'),
    'TX123': ('TX123_2023_12_21_1', 'TX123_2024_01_02_1'),
    'DR10':  ('DR10_2022_07_12_1',  'DR10_2022_07_19_1'),
    'DR15':  ('DR15_2022_10_09_1',  'DR15_2022_10_19_1'),
    'TX85':  ('TX85_2022_06_14_1',  'TX85_2022_06_17_1'),
}

BEH_FILES = {
    ('sup', 'bef'):   'Beh_sup_train1_before_learning.npy',
    ('sup', 'aft'):   'Beh_sup_train1_after_learning.npy',
    ('unsup', 'bef'): 'Beh_unsup_train1_before_learning.npy',
    ('unsup', 'aft'): 'Beh_unsup_train1_after_learning.npy',
}


def get_condition_prefix(mouse_name):
    return 'sup' if mouse_name in supervised_mice else 'unsup'


# ------------------------------------------------------------
# Rui's function - unchanged
# ------------------------------------------------------------
def calculate_pair_dprime_and_speed_window_history(
    proj_data, stim_fr, pos_fr, VR_move, run_speed_fr, name_stim1, name_stim2, window_size=20
):
    n_neurons, n_frames = proj_data.shape

    stim_fr = stim_fr[:n_frames]
    pos_fr = pos_fr[:n_frames]
    VR_move = VR_move[:n_frames]
    run_speed_fr = run_speed_fr[:n_frames]

    pos_diff = np.diff(pos_fr, prepend=pos_fr[0])
    stim_fr_changed = (stim_fr != np.roll(stim_fr, 1))
    stim_fr_changed[0] = False
    trial_starts = (pos_diff < -10) | stim_fr_changed
    trial_indices = np.cumsum(trial_starts)

    unique_trials = np.unique(trial_indices)

    stim1_trials_frames = []
    stim2_trials_frames = []
    stim1_trials_speed = []
    stim2_trials_speed = []

    for t in unique_trials:
        t_mask = (trial_indices == t) & (pos_fr < 40) & VR_move
        if np.sum(t_mask) < 3:
            continue

        t_stim_name = stim_fr[t_mask][0]
        trial_data = proj_data[:, t_mask]
        trial_speed = run_speed_fr[t_mask]

        if t_stim_name == name_stim1:
            stim1_trials_frames.append(trial_data)
            stim1_trials_speed.append(trial_speed)
        elif t_stim_name == name_stim2:
            stim2_trials_frames.append(trial_data)
            stim2_trials_speed.append(trial_speed)

    n_pairs = min(len(stim1_trials_frames), len(stim2_trials_frames))
    print(f"\n[Pair info:]")
    print(f"├─ Stim1 ({name_stim1}) has {len(stim1_trials_frames)} trials")
    print(f"├─ Stim2 ({name_stim2}) has {len(stim2_trials_frames)} trials")
    print(f"└─ There are {n_pairs} pairs of stimuli.")

    if n_pairs < window_size:
        print(f"  SKIPPING: not enough pairs ({n_pairs}) for window size ({window_size})")
        return None, None

    dprime_history = []
    mean_speed_history = []
    n_windows = n_pairs - window_size + 1
    print(f"[Sliding Window info] Window Size = {window_size} | Will have {n_windows} windows for tracking")

    for w in range(n_windows):
        window_leaf_data = np.hstack(stim1_trials_frames[w: w + window_size])
        window_circle_data = np.hstack(stim2_trials_frames[w: w + window_size])
        window_leaf_speed = np.hstack(stim1_trials_speed[w: w + window_size])
        window_circle_speed = np.hstack(stim2_trials_speed[w: w + window_size])

        u_leaf = np.nanmean(window_leaf_data, axis=1)
        u_circle = np.nanmean(window_circle_data, axis=1)
        sig_leaf = np.nanstd(window_leaf_data, axis=1)
        sig_circle = np.nanstd(window_circle_data, axis=1)

        denominator = sig_leaf + sig_circle + 1e-8
        dp_w = 2 * (u_leaf - u_circle) / denominator
        dprime_history.append(dp_w)

        all_speed_in_window = np.concatenate([window_leaf_speed, window_circle_speed])
        avg_speed_w = np.nanmean(all_speed_in_window)
        mean_speed_history.append(avg_speed_w)

    dprime_history = np.column_stack(dprime_history)
    dprime_history = np.nan_to_num(dprime_history, nan=0.0, posinf=0.0, neginf=0.0)
    mean_speed_history = np.array(mean_speed_history)

    return dprime_history, mean_speed_history


# ------------------------------------------------------------
# Loop over ALL 13 mice, both before and after sessions
# ------------------------------------------------------------
results = {}  # results['VR2_bef'] = (dprime, speed), results['VR2_aft'] = (...), etc.

for mouse_name, (session_bef, session_aft) in mice_sessions.items():
    condition = get_condition_prefix(mouse_name)

    for tag, session_id in [('bef', session_bef), ('aft', session_aft)]:
        result_key = f"{mouse_name}_{tag}"
        print(f"\n{'=' * 60}")
        print(f"Processing: {result_key}  (condition: {condition}_{tag}, session: {session_id})")
        print('=' * 60)

        try:
            beh_filename = BEH_FILES[(condition, tag)]
            beh = np.load(os.path.join(root, beh_filename), allow_pickle=1).item()[session_id]

            stim_id = beh['stim_id']
            uniqW, WallN = beh['UniqWalls'], beh['WallName']
            stim_fr = beh['ft_WallID']
            pos_fr = beh['ft_Pos']
            VR_move = beh['ft_move'] > 0
            RunSpeed_fr = beh['ft_RunSpeed']

            name_stim1 = uniqW[stim_id == 2][0]
            name_stim2 = uniqW[stim_id == 0][0]

            svd_dec = np.load(os.path.join(root, session_id + '_SVD_dec.npy'), allow_pickle=1).item()
            proj_data = svd_dec['U'].T @ svd_dec['V']

            dprime_history, speed_history = calculate_pair_dprime_and_speed_window_history(
                proj_data, stim_fr, pos_fr, VR_move, RunSpeed_fr, name_stim1, name_stim2, window_size=20
            )

            results[result_key] = (dprime_history, speed_history)

            del svd_dec, proj_data, beh
            gc.collect()

        except Exception as error:
            print(f"  SKIPPED {result_key}: {type(error).__name__}: {error}")
            continue

print(f"\n\nDone! Processed {len(results)} sessions out of {len(mice_sessions) * 2} expected.")
print("Access results like: results['VR2_bef'][0]  <- d' history")
print("                     results['VR2_bef'][1]  <- speed history")

# ------------------------------------------------------------
# Save summary stats (matching Rui's CSV format) for all 26 sessions
# ------------------------------------------------------------
import pandas as pd
from scipy import stats as scipy_stats

DP_THRESHOLD = 0.3
output_folder = root  # saves alongside your data; change if you'd like elsewhere

for mouse_name, (session_bef, session_aft) in mice_sessions.items():
    condition = get_condition_prefix(mouse_name)

    for tag, session_id in [('bef', session_bef), ('aft', session_aft)]:
        result_key = f"{mouse_name}_{tag}"

        if result_key not in results:
            print(f"Skipping {result_key} - no results found (was likely skipped earlier)")
            continue

        dprime_history, speed_history = results[result_key]

        if dprime_history is None:
            print(f"Skipping {result_key} - dprime history is empty")
            continue

        n_windows = dprime_history.shape[1]
        rows = []

        for w in range(n_windows):
            dp_window = dprime_history[:, w]  # every neuron's d' for this window

            rows.append({
                'Window_ID': w + 1,
                'Median': float(np.median(dp_window)),
                'Std_Dev': float(np.std(dp_window)),
                'Skewness': float(scipy_stats.skew(dp_window)),
                'Fraction_Leaf_Selective_gte_0.3': float(np.mean(dp_window >= DP_THRESHOLD)),
                'Fraction_Circle_Selective_lte_-0.3': float(np.mean(dp_window <= -DP_THRESHOLD)),
                'Running_Speed': float(speed_history[w]),
                'Mouse_ID': mouse_name,
                'Condition': f"{condition}_{tag}",
            })

        df = pd.DataFrame(rows)
        out_path = os.path.join(output_folder, f"{mouse_name}_{session_id if False else ''}".strip('_'))
        # build filename matching Rui's convention
        session_id_used = session_bef if tag == 'bef' else session_aft
        out_path = os.path.join(output_folder, f"{session_id_used}_dprime_stats_{condition}_{tag}_w20.csv")

        df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}  ({len(df)} windows)")

print("\nAll CSVs saved.")

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Creates 4 animated GIFs, in this order:
1. Supervised, Before Learning
2. Unsupervised, Before Learning
3. Supervised, After Learning
4. Unsupervised, After Learning

Each shows the pooled d' distribution across all mice in that
group/condition, for the first 100 windows (padded with NaN if a
mouse's session is shorter than 100 windows).
"""

from pathlib import Path
import os
import gc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.stats import gaussian_kde

# ------------------------------------------------------------
# Your data folder
# ------------------------------------------------------------
root = Path(r"C:\Users\tasha\OneDrive\Desktop\Neuromatch Academy")

N_WINDOWS_TARGET = 100

supervised_mice_bef = {
    'VR2': 'VR2_2021_03_20_1', 'TX60': 'TX60_2021_04_10_1',
    'TX108': 'TX108_2023_03_13_1', 'TX109': 'TX109_2023_03_27_1',
}
supervised_mice_aft = {
    'VR2': 'VR2_2021_04_06_1', 'TX60': 'TX60_2021_05_04_1',
    'TX108': 'TX108_2023_03_22_1', 'TX109': 'TX109_2023_04_07_1',
}
unsupervised_mice_bef = {
    'TX83': 'TX83_2022_08_17_1', 'TX88': 'TX88_2022_06_13_2',
    'TX105': 'TX105_2022_10_08_2', 'TX104': 'TX104_2022_10_12_1',
    'TX119': 'TX119_2023_12_14_1', 'TX123': 'TX123_2023_12_21_1',
    'DR10': 'DR10_2022_07_12_1', 'DR15': 'DR15_2022_10_09_1',
    'TX85': 'TX85_2022_06_14_1',
}
unsupervised_mice_aft = {
    'TX83': 'TX83_2022_08_29_1', 'TX88': 'TX88_2022_06_17_2',
    'TX105': 'TX105_2022_10_19_2', 'TX104': 'TX104_2022_10_20_1',
    'TX119': 'TX119_2023_12_23_1', 'TX123': 'TX123_2024_01_02_1',
    'DR10': 'DR10_2022_07_19_1', 'DR15': 'DR15_2022_10_19_1',
    'TX85': 'TX85_2022_06_17_1',
}

# order: both "before" conditions first, then both "after" conditions
CONDITIONS = [
    ('sup_bef',   supervised_mice_bef,   'Beh_sup_train1_before_learning.npy',   'Supervised, Before Learning'),
    ('unsup_bef', unsupervised_mice_bef, 'Beh_unsup_train1_before_learning.npy', 'Unsupervised, Before Learning'),
    ('sup_aft',   supervised_mice_aft,   'Beh_sup_train1_after_learning.npy',    'Supervised, After Learning'),
    ('unsup_aft', unsupervised_mice_aft, 'Beh_unsup_train1_after_learning.npy',  'Unsupervised, After Learning'),
]


# ------------------------------------------------------------
# Rui's original sliding-window d' function (unchanged)
# ------------------------------------------------------------
def calculate_single_pair_dprime_history(proj_data, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=20):
    n_neurons, n_frames = proj_data.shape

    stim_fr = stim_fr[:n_frames]
    pos_fr = pos_fr[:n_frames]
    VR_move = VR_move[:n_frames]

    pos_diff = np.diff(pos_fr, prepend=pos_fr[0])
    stim_fr_changed = (stim_fr != np.roll(stim_fr, 1))
    stim_fr_changed[0] = False
    trial_starts = (pos_diff < -10) | stim_fr_changed
    trial_indices = np.cumsum(trial_starts)

    unique_trials = np.unique(trial_indices)

    stim1_trials_frames = []
    stim2_trials_frames = []

    for t in unique_trials:
        t_mask = (trial_indices == t) & (pos_fr < 40) & VR_move
        if np.sum(t_mask) < 3:
            continue

        t_stim_name = stim_fr[t_mask][0]
        trial_data = proj_data[:, t_mask]

        if t_stim_name == name_stim1:
            stim1_trials_frames.append(trial_data)
        elif t_stim_name == name_stim2:
            stim2_trials_frames.append(trial_data)

    n_pairs = min(len(stim1_trials_frames), len(stim2_trials_frames))
    print(f"    Stim1 has {len(stim1_trials_frames)} trials, Stim2 has {len(stim2_trials_frames)} trials, {n_pairs} pairs")

    if n_pairs < window_size:
        print(f"    SKIPPING: not enough pairs ({n_pairs}) for window size ({window_size})")
        return None

    dprime_history = []
    n_windows = n_pairs - window_size + 1

    for w in range(n_windows):
        window_leaf_data = np.hstack(stim1_trials_frames[w: w + window_size])
        window_circle_data = np.hstack(stim2_trials_frames[w: w + window_size])

        u_leaf = np.nanmean(window_leaf_data, axis=1)
        u_circle = np.nanmean(window_circle_data, axis=1)
        sig_leaf = np.nanstd(window_leaf_data, axis=1)
        sig_circle = np.nanstd(window_circle_data, axis=1)

        denominator = sig_leaf + sig_circle + 1e-8
        dp_w = 2 * (u_leaf - u_circle) / denominator
        dprime_history.append(dp_w)

    dprime_history = np.column_stack(dprime_history)
    dprime_history = np.nan_to_num(dprime_history, nan=0.0, posinf=0.0, neginf=0.0)

    return dprime_history


def load_and_compute(session_id, beh_filename, window_size=20):
    beh = np.load(os.path.join(root, beh_filename), allow_pickle=1).item()[session_id]

    stim_id = beh['stim_id']
    uniqW, WallN = beh['UniqWalls'], beh['WallName']
    stim_fr = beh['ft_WallID']
    pos_fr = beh['ft_Pos']
    VR_move = beh['ft_move'] > 0

    name_stim1 = uniqW[stim_id == 2][0]
    name_stim2 = uniqW[stim_id == 0][0]

    svd_dec = np.load(os.path.join(root, session_id + '_SVD_dec.npy'), allow_pickle=1).item()
    # float32 keeps memory use as low as possible
    proj_data = (svd_dec['U'].T @ svd_dec['V']).astype(np.float32)

    # free the raw SVD components immediately, before the d' calculation runs
    del svd_dec
    gc.collect()

    dprime_history = calculate_single_pair_dprime_history(
        proj_data, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=window_size
    )

    del proj_data, beh
    gc.collect()

    return dprime_history


def pad_to_target(dprime_history, target_windows):
    n_neurons, n_windows = dprime_history.shape
    if n_windows >= target_windows:
        return dprime_history[:, :target_windows]
    else:
        pad_width = target_windows - n_windows
        nan_pad = np.full((n_neurons, pad_width), np.nan)
        return np.hstack([dprime_history, nan_pad])


def process_group(mice_dict, beh_filename, group_name):
    print(f"\n{'=' * 60}")
    print(f"Processing group: {group_name}")
    print('=' * 60)

    all_mice_padded = []

    for mouse_name, session_id in mice_dict.items():
        print(f"\n  Mouse: {mouse_name} ({session_id})")
        try:
            dprime_history = load_and_compute(session_id, beh_filename, window_size=20)
            if dprime_history is None:
                print(f"    Skipping {mouse_name} - not enough data")
                continue

            padded = pad_to_target(dprime_history, N_WINDOWS_TARGET)
            all_mice_padded.append(padded)
            print(f"    {mouse_name}: had {dprime_history.shape[1]} windows, padded/trimmed to {N_WINDOWS_TARGET}")

        except Exception as error:
            print(f"    SKIPPED {mouse_name}: {type(error).__name__}: {error}")
            continue

    pooled_matrix = np.vstack(all_mice_padded)
    print(f"\n  Pooled matrix shape for {group_name}: {pooled_matrix.shape}  (neurons x windows)")

    return pooled_matrix


# ------------------------------------------------------------
# Animation function - fixed x-axis (-1.5 to 1.5), stable y-axis,
# no "(n=...)" in title
# ------------------------------------------------------------
def animate_dprime_distribution(dprime_matrix, fps=10, save_path="dprime_evolution.gif", title_prefix=""):
    n_neurons, n_windows = dprime_matrix.shape
    print(f"Preparing animation: {n_neurons} pooled neurons, {n_windows} windows...")

    # --- X-AXIS: fixed at -1.5 to 1.5 ---
    x_min, x_max = -1.5, 1.5
    x_eval = np.linspace(x_min, x_max, 300)

    # --- Y-AXIS: precompute tallest bar across all frames with real data ---
    print("  Calculating stable y-axis limit across all frames...")
    max_density = 0
    for frame in range(n_windows):
        current_data = dprime_matrix[:, frame]
        current_data = current_data[~np.isnan(current_data)]
        if current_data.size == 0:
            print(f"    Window {frame + 1}: NO real data (0 neurons)")
            continue
        counts, _ = np.histogram(current_data, bins=60, range=(x_min, x_max), density=True)
        max_density = max(max_density, counts.max())
    y_max = max_density * 1.15

    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

    def update(frame):
        ax.clear()
        current_data = dprime_matrix[:, frame]
        current_data = current_data[~np.isnan(current_data)]

        if current_data.size == 0:
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(0, y_max)
            ax.set_title(f"{title_prefix} | Window {frame + 1}/{n_windows} (no data)", fontsize=12)
            return

        ax.hist(current_data, bins=60, range=(x_min, x_max),
                density=True, alpha=0.4, color='#4A90E2', label='Neurons Hist')

        if len(current_data) > 1000:
            sample_data = np.random.choice(current_data, size=1000, replace=False)
        else:
            sample_data = current_data

        try:
            kde = gaussian_kde(sample_data)
            ax.plot(x_eval, kde(x_eval), color='#D0021B', lw=2.5, label='KDE Fit')
        except Exception:
            pass

        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(0, y_max)
        ax.set_xlabel("d-prime value", fontsize=11)
        ax.set_ylabel("Density", fontsize=11)
        ax.set_title(f"{title_prefix} | Window {frame + 1}/{n_windows}", fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(axis='y', alpha=0.3)

    anim = FuncAnimation(fig, update, frames=n_windows, interval=1000 / fps)

    print(f"Saving {save_path}, please wait...")
    writer = PillowWriter(fps=fps)
    anim.save(save_path, writer=writer)
    plt.close()
    print("Finished!")


# ------------------------------------------------------------
# Run all 4 conditions, in order: sup_bef, unsup_bef, sup_aft, unsup_aft
# ------------------------------------------------------------
for condition_key, mice_dict, beh_filename, title in CONDITIONS:
    pooled = process_group(mice_dict, beh_filename, title)
    animate_dprime_distribution(
        pooled, fps=12,
        save_path=str(root / f"dprime_GROUP_{condition_key}_first100windows.gif"),
        title_prefix=title
    )

print("\n\nAll 4 group-pooled animations complete!")

"""
Created on Wed Jul 22 21:36:12 2026

@author: tasha
"""

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Computes d' for mHV (Medial Higher Visual Area) NEURONS ONLY,
for all 13 mice, before and after learning - 26 sessions total.
Saves one CSV per session.

Same memory-safe approach as the V1 version.
"""

from pathlib import Path
import os
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

# ------------------------------------------------------------
# Your data folder
# ------------------------------------------------------------
root = Path(r"C:\Users\tasha\OneDrive\Desktop\Neuromatch Academy")

supervised_mice = ['VR2', 'TX60', 'TX108', 'TX109']
unsupervised_mice = ['TX83', 'TX88', 'TX105', 'TX104', 'TX119', 'TX123', 'DR10', 'DR15', 'TX85']

mice_sessions = {
    'VR2':   ('VR2_2021_03_20_1',   'VR2_2021_04_06_1'),
    'TX60':  ('TX60_2021_04_10_1',  'TX60_2021_05_04_1'),
    'TX108': ('TX108_2023_03_13_1', 'TX108_2023_03_22_1'),
    'TX109': ('TX109_2023_03_27_1', 'TX109_2023_04_07_1'),
    'TX83':  ('TX83_2022_08_17_1',  'TX83_2022_08_29_1'),
    'TX88':  ('TX88_2022_06_13_2',  'TX88_2022_06_17_2'),
    'TX105': ('TX105_2022_10_08_2', 'TX105_2022_10_19_2'),
    'TX104': ('TX104_2022_10_12_1', 'TX104_2022_10_20_1'),
    'TX119': ('TX119_2023_12_14_1', 'TX119_2023_12_23_1'),
    'TX123': ('TX123_2023_12_21_1', 'TX123_2024_01_02_1'),
    'DR10':  ('DR10_2022_07_12_1',  'DR10_2022_07_19_1'),
    'DR15':  ('DR15_2022_10_09_1',  'DR15_2022_10_19_1'),
    'TX85':  ('TX85_2022_06_14_1',  'TX85_2022_06_17_1'),
}

BEH_FILES = {
    ('sup', 'bef'):   'Beh_sup_train1_before_learning.npy',
    ('sup', 'aft'):   'Beh_sup_train1_after_learning.npy',
    ('unsup', 'bef'): 'Beh_unsup_train1_before_learning.npy',
    ('unsup', 'aft'): 'Beh_unsup_train1_after_learning.npy',
}

DP_THRESHOLD = 0.3


def get_condition_prefix(mouse_name):
    return 'sup' if mouse_name in supervised_mice else 'unsup'


# ------------------------------------------------------------
# Rui's area mapping function - exact copy
# ------------------------------------------------------------
def neu_area_ID(iarea):
    area_name = ['V1', 'mHV', 'lHV', 'aHV']
    idx = {}
    for ar in area_name:
        if ar == 'V1':
            idx[ar] = iarea == 8
        elif ar == 'mHV':
            idx[ar] = (iarea == 0) | (iarea == 1) | (iarea == 2) | (iarea == 9)
        elif ar == 'lHV':
            idx[ar] = (iarea == 5) | (iarea == 6)
        elif ar == 'aHV':
            idx[ar] = (iarea == 3) | (iarea == 4)
    return idx


# ------------------------------------------------------------
# Memory-safe d' calculation
# ------------------------------------------------------------
def calculate_dprime_history_lazy(U_subset, V, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=20):
    n_frames = V.shape[1]

    stim_fr = stim_fr[:n_frames]
    pos_fr = pos_fr[:n_frames]
    VR_move = VR_move[:n_frames]

    pos_diff = np.diff(pos_fr, prepend=pos_fr[0])
    stim_fr_changed = (stim_fr != np.roll(stim_fr, 1))
    stim_fr_changed[0] = False
    trial_starts = (pos_diff < -10) | stim_fr_changed
    trial_indices = np.cumsum(trial_starts)

    unique_trials = np.unique(trial_indices)

    stim1_trials_masks = []
    stim2_trials_masks = []

    for t in unique_trials:
        t_mask = (trial_indices == t) & (pos_fr < 40) & VR_move
        if np.sum(t_mask) < 3:
            continue

        t_stim_name = stim_fr[t_mask][0]

        if t_stim_name == name_stim1:
            stim1_trials_masks.append(t_mask)
        elif t_stim_name == name_stim2:
            stim2_trials_masks.append(t_mask)

    n_pairs = min(len(stim1_trials_masks), len(stim2_trials_masks))
    print(f"    Stim1 has {len(stim1_trials_masks)} trials, Stim2 has {len(stim2_trials_masks)} trials, {n_pairs} pairs")

    if n_pairs < window_size:
        print("    SKIPPING: not enough pairs")
        return None

    dprime_history = []
    n_windows = n_pairs - window_size + 1

    for w in range(n_windows):
        leaf_mask = np.any(stim1_trials_masks[w: w + window_size], axis=0)
        circle_mask = np.any(stim2_trials_masks[w: w + window_size], axis=0)

        window_leaf_data = U_subset.T @ V[:, leaf_mask]
        window_circle_data = U_subset.T @ V[:, circle_mask]

        u_leaf = np.nanmean(window_leaf_data, axis=1)
        u_circle = np.nanmean(window_circle_data, axis=1)
        sig_leaf = np.nanstd(window_leaf_data, axis=1)
        sig_circle = np.nanstd(window_circle_data, axis=1)

        denominator = sig_leaf + sig_circle + 1e-8
        dp_w = 2 * (u_leaf - u_circle) / denominator
        dprime_history.append(dp_w)

        del window_leaf_data, window_circle_data

    dprime_history = np.column_stack(dprime_history)
    dprime_history = np.nan_to_num(dprime_history, nan=0.0, posinf=0.0, neginf=0.0)

    return dprime_history


# ------------------------------------------------------------
# Load one session's data, filter to mHV neurons, compute d'
# ------------------------------------------------------------
def process_session_mhv(session_id, beh_filename):
    beh = np.load(os.path.join(root, beh_filename), allow_pickle=1).item()[session_id]

    stim_id = beh['stim_id']
    uniqW = beh['UniqWalls']
    stim_fr = beh['ft_WallID']
    pos_fr = beh['ft_Pos']
    VR_move = beh['ft_move'] > 0

    name_stim1 = uniqW[stim_id == 2][0]
    name_stim2 = uniqW[stim_id == 0][0]

    # load area info for this session (trans.npz)
    trans_filename = session_id.rsplit('_', 1)[0] + '_trans.npz'
    trans = np.load(os.path.join(root, trans_filename), allow_pickle=True)
    area_idx = neu_area_ID(trans['iarea'])
    mhv_mask = area_idx['mHV']

    del trans
    gc.collect()

    # load SVD components
    svd_dec = np.load(os.path.join(root, session_id + '_SVD_dec.npy'), allow_pickle=1).item()
    U = svd_dec['U'].astype(np.float32)
    V = svd_dec['V'].astype(np.float32)

    del svd_dec
    gc.collect()

    if mhv_mask.shape[0] != U.shape[1]:
        print(f"    WARNING: mhv_mask length ({mhv_mask.shape[0]}) doesn't match U's neuron count ({U.shape[1]})")

    U_mhv = U[:, mhv_mask]  # keep only mHV neurons
    print(f"    mHV neurons: {U_mhv.shape[1]} out of {U.shape[1]} total")

    del U
    gc.collect()

    dprime_history = calculate_dprime_history_lazy(
        U_mhv, V, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=20
    )

    del U_mhv, V, beh
    gc.collect()

    return dprime_history


# ------------------------------------------------------------
# Loop over all 13 mice, both sessions, save CSVs
# ------------------------------------------------------------
results = {}

for mouse_name, (session_bef, session_aft) in mice_sessions.items():
    condition = get_condition_prefix(mouse_name)

    for tag, session_id in [('bef', session_bef), ('aft', session_aft)]:
        result_key = f"{mouse_name}_{tag}"
        print(f"\n{'=' * 60}")
        print(f"Processing mHV: {result_key}  (session: {session_id})")
        print('=' * 60)

        try:
            beh_filename = BEH_FILES[(condition, tag)]
            dprime_history = process_session_mhv(session_id, beh_filename)
            results[result_key] = dprime_history

        except Exception as error:
            print(f"  SKIPPED {result_key}: {type(error).__name__}: {error}")
            continue

print(f"\n\nDone processing! {len(results)} sessions out of {len(mice_sessions) * 2} expected.")

# ------------------------------------------------------------
# Save summary stats as CSVs (mHV only)
# ------------------------------------------------------------
output_folder = root

for mouse_name, (session_bef, session_aft) in mice_sessions.items():
    condition = get_condition_prefix(mouse_name)

    for tag, session_id in [('bef', session_bef), ('aft', session_aft)]:
        result_key = f"{mouse_name}_{tag}"

        if result_key not in results or results[result_key] is None:
            print(f"Skipping {result_key} - no mHV results")
            continue

        dprime_history = results[result_key]
        n_windows = dprime_history.shape[1]
        rows = []

        for w in range(n_windows):
            dp_window = dprime_history[:, w]
            rows.append({
                'Window_ID': w + 1,
                'Median': float(np.median(dp_window)),
                'Std_Dev': float(np.std(dp_window)),
                'Skewness': float(scipy_stats.skew(dp_window)),
                'Fraction_Leaf_Selective_gte_0.3': float(np.mean(dp_window >= DP_THRESHOLD)),
                'Fraction_Circle_Selective_lte_-0.3': float(np.mean(dp_window <= -DP_THRESHOLD)),
                'Mouse_ID': mouse_name,
                'Condition': f"{condition}_{tag}",
                'Area': 'mHV',
            })

        df = pd.DataFrame(rows)
        session_id_used = session_bef if tag == 'bef' else session_aft
        out_path = os.path.join(output_folder, f"{session_id_used}_dprime_stats_mHV_{condition}_{tag}_w20.csv")
        df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}  ({len(df)} windows)")

print("\nAll mHV CSVs saved.")

"""
Created on Wed Jul 22 20:41:55 2026

@author: tasha
"""

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Computes d' for V1 NEURONS ONLY, for all 13 mice, before and after
learning - 26 sessions total. Saves one CSV per session.

Uses the memory-safe approach (small chunks, not the full matrix),
plus filters to V1 neurons before reconstruction to keep memory low.
"""

from pathlib import Path
import os
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

# ------------------------------------------------------------
# Your data folder
# ------------------------------------------------------------
root = Path(r"C:\Users\tasha\OneDrive\Desktop\Neuromatch Academy")

supervised_mice = ['VR2', 'TX60', 'TX108', 'TX109']
unsupervised_mice = ['TX83', 'TX88', 'TX105', 'TX104', 'TX119', 'TX123', 'DR10', 'DR15', 'TX85']

mice_sessions = {
    'VR2':   ('VR2_2021_03_20_1',   'VR2_2021_04_06_1'),
    'TX60':  ('TX60_2021_04_10_1',  'TX60_2021_05_04_1'),
    'TX108': ('TX108_2023_03_13_1', 'TX108_2023_03_22_1'),
    'TX109': ('TX109_2023_03_27_1', 'TX109_2023_04_07_1'),
    'TX83':  ('TX83_2022_08_17_1',  'TX83_2022_08_29_1'),
    'TX88':  ('TX88_2022_06_13_2',  'TX88_2022_06_17_2'),
    'TX105': ('TX105_2022_10_08_2', 'TX105_2022_10_19_2'),
    'TX104': ('TX104_2022_10_12_1', 'TX104_2022_10_20_1'),
    'TX119': ('TX119_2023_12_14_1', 'TX119_2023_12_23_1'),
    'TX123': ('TX123_2023_12_21_1', 'TX123_2024_01_02_1'),
    'DR10':  ('DR10_2022_07_12_1',  'DR10_2022_07_19_1'),
    'DR15':  ('DR15_2022_10_09_1',  'DR15_2022_10_19_1'),
    'TX85':  ('TX85_2022_06_14_1',  'TX85_2022_06_17_1'),
}

BEH_FILES = {
    ('sup', 'bef'):   'Beh_sup_train1_before_learning.npy',
    ('sup', 'aft'):   'Beh_sup_train1_after_learning.npy',
    ('unsup', 'bef'): 'Beh_unsup_train1_before_learning.npy',
    ('unsup', 'aft'): 'Beh_unsup_train1_after_learning.npy',
}

DP_THRESHOLD = 0.3


def get_condition_prefix(mouse_name):
    return 'sup' if mouse_name in supervised_mice else 'unsup'


# ------------------------------------------------------------
# Rui's area mapping function - exact copy
# ------------------------------------------------------------
def neu_area_ID(iarea):
    area_name = ['V1', 'mHV', 'lHV', 'aHV']
    idx = {}
    for ar in area_name:
        if ar == 'V1':
            idx[ar] = iarea == 8
        elif ar == 'mHV':
            idx[ar] = (iarea == 0) | (iarea == 1) | (iarea == 2) | (iarea == 9)
        elif ar == 'lHV':
            idx[ar] = (iarea == 5) | (iarea == 6)
        elif ar == 'aHV':
            idx[ar] = (iarea == 3) | (iarea == 4)
    return idx


# ------------------------------------------------------------
# Memory-safe d' calculation - only reconstructs small window
# slices at a time, and only for the neurons we care about
# ------------------------------------------------------------
def calculate_dprime_history_lazy(U_subset, V, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=20):
    n_frames = V.shape[1]

    stim_fr = stim_fr[:n_frames]
    pos_fr = pos_fr[:n_frames]
    VR_move = VR_move[:n_frames]

    pos_diff = np.diff(pos_fr, prepend=pos_fr[0])
    stim_fr_changed = (stim_fr != np.roll(stim_fr, 1))
    stim_fr_changed[0] = False
    trial_starts = (pos_diff < -10) | stim_fr_changed
    trial_indices = np.cumsum(trial_starts)

    unique_trials = np.unique(trial_indices)

    stim1_trials_masks = []
    stim2_trials_masks = []

    for t in unique_trials:
        t_mask = (trial_indices == t) & (pos_fr < 40) & VR_move
        if np.sum(t_mask) < 3:
            continue

        t_stim_name = stim_fr[t_mask][0]

        if t_stim_name == name_stim1:
            stim1_trials_masks.append(t_mask)
        elif t_stim_name == name_stim2:
            stim2_trials_masks.append(t_mask)

    n_pairs = min(len(stim1_trials_masks), len(stim2_trials_masks))
    print(f"    Stim1 has {len(stim1_trials_masks)} trials, Stim2 has {len(stim2_trials_masks)} trials, {n_pairs} pairs")

    if n_pairs < window_size:
        print("    SKIPPING: not enough pairs")
        return None

    dprime_history = []
    n_windows = n_pairs - window_size + 1

    for w in range(n_windows):
        leaf_mask = np.any(stim1_trials_masks[w: w + window_size], axis=0)
        circle_mask = np.any(stim2_trials_masks[w: w + window_size], axis=0)

        window_leaf_data = U_subset.T @ V[:, leaf_mask]
        window_circle_data = U_subset.T @ V[:, circle_mask]

        u_leaf = np.nanmean(window_leaf_data, axis=1)
        u_circle = np.nanmean(window_circle_data, axis=1)
        sig_leaf = np.nanstd(window_leaf_data, axis=1)
        sig_circle = np.nanstd(window_circle_data, axis=1)

        denominator = sig_leaf + sig_circle + 1e-8
        dp_w = 2 * (u_leaf - u_circle) / denominator
        dprime_history.append(dp_w)

        del window_leaf_data, window_circle_data

    dprime_history = np.column_stack(dprime_history)
    dprime_history = np.nan_to_num(dprime_history, nan=0.0, posinf=0.0, neginf=0.0)

    return dprime_history


# ------------------------------------------------------------
# Load one session's data, filter to V1 neurons, compute d'
# ------------------------------------------------------------
def process_session_v1(session_id, beh_filename):
    beh = np.load(os.path.join(root, beh_filename), allow_pickle=1).item()[session_id]

    stim_id = beh['stim_id']
    uniqW = beh['UniqWalls']
    stim_fr = beh['ft_WallID']
    pos_fr = beh['ft_Pos']
    VR_move = beh['ft_move'] > 0

    name_stim1 = uniqW[stim_id == 2][0]
    name_stim2 = uniqW[stim_id == 0][0]

    # load area info for this session (trans.npz)
    trans_filename = session_id.rsplit('_', 1)[0] + '_trans.npz'
    trans = np.load(os.path.join(root, trans_filename), allow_pickle=True)
    area_idx = neu_area_ID(trans['iarea'])
    v1_mask = area_idx['V1']

    del trans
    gc.collect()

    # load SVD components
    svd_dec = np.load(os.path.join(root, session_id + '_SVD_dec.npy'), allow_pickle=1).item()
    U = svd_dec['U'].astype(np.float32)  # shape: (components, neurons)
    V = svd_dec['V'].astype(np.float32)  # shape: (components, frames)

    del svd_dec
    gc.collect()

    if v1_mask.shape[0] != U.shape[1]:
        print(f"    WARNING: v1_mask length ({v1_mask.shape[0]}) doesn't match U's neuron count ({U.shape[1]})")

    U_v1 = U[:, v1_mask]  # keep only V1 neurons
    print(f"    V1 neurons: {U_v1.shape[1]} out of {U.shape[1]} total")

    del U
    gc.collect()

    dprime_history = calculate_dprime_history_lazy(
        U_v1, V, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=20
    )

    del U_v1, V, beh
    gc.collect()

    return dprime_history


# ------------------------------------------------------------
# Loop over all 13 mice, both sessions, save CSVs
# ------------------------------------------------------------
results = {}

for mouse_name, (session_bef, session_aft) in mice_sessions.items():
    condition = get_condition_prefix(mouse_name)

    for tag, session_id in [('bef', session_bef), ('aft', session_aft)]:
        result_key = f"{mouse_name}_{tag}"
        print(f"\n{'=' * 60}")
        print(f"Processing V1: {result_key}  (session: {session_id})")
        print('=' * 60)

        try:
            beh_filename = BEH_FILES[(condition, tag)]
            dprime_history = process_session_v1(session_id, beh_filename)
            results[result_key] = dprime_history

        except Exception as error:
            print(f"  SKIPPED {result_key}: {type(error).__name__}: {error}")
            continue

print(f"\n\nDone processing! {len(results)} sessions out of {len(mice_sessions) * 2} expected.")

# ------------------------------------------------------------
# Save summary stats as CSVs (V1 only)
# ------------------------------------------------------------
output_folder = root

for mouse_name, (session_bef, session_aft) in mice_sessions.items():
    condition = get_condition_prefix(mouse_name)

    for tag, session_id in [('bef', session_bef), ('aft', session_aft)]:
        result_key = f"{mouse_name}_{tag}"

        if result_key not in results or results[result_key] is None:
            print(f"Skipping {result_key} - no V1 results")
            continue

        dprime_history = results[result_key]
        n_windows = dprime_history.shape[1]
        rows = []

        for w in range(n_windows):
            dp_window = dprime_history[:, w]
            rows.append({
                'Window_ID': w + 1,
                'Median': float(np.median(dp_window)),
                'Std_Dev': float(np.std(dp_window)),
                'Skewness': float(scipy_stats.skew(dp_window)),
                'Fraction_Leaf_Selective_gte_0.3': float(np.mean(dp_window >= DP_THRESHOLD)),
                'Fraction_Circle_Selective_lte_-0.3': float(np.mean(dp_window <= -DP_THRESHOLD)),
                'Mouse_ID': mouse_name,
                'Condition': f"{condition}_{tag}",
                'Area': 'V1',
            })

        df = pd.DataFrame(rows)
        session_id_used = session_bef if tag == 'bef' else session_aft
        out_path = os.path.join(output_folder, f"{session_id_used}_dprime_stats_V1_{condition}_{tag}_w20.csv")
        df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}  ({len(df)} windows)")

print("\nAll V1 CSVs saved.")
